# Semana 13 — Ingeniería de características
Juan Esteban Camacho Castro, Juan Diego Serpa Ruiz

## Carga del dataset integrado

In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("S12_PD_Camacho_Serpa_DatasetIntegrado.csv", low_memory=False)

In [4]:
df.shape

(599000, 64)

## Fecha de referencia del análisis
Usamos una **fecha fija documentada** para que las features se puedan reproducir sin cambiar según el día en que se corra el notebook. La fecha elegida es `2027-12-29`, un día después del pedido más tardío del dataset (2027-12-28), para que todos los `dias_desde_pedido` queden positivos y con sentido de negocio.

In [5]:
fecha_corte = pd.Timestamp("2027-12-29")

### Conversión de las columnas de fecha

In [6]:
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"], errors="coerce")

In [7]:
df["fecha_inscripcion_club"] = pd.to_datetime(df["fecha_inscripcion_club"], errors="coerce")

## Ingeniería de características

### Familia 1: Variables derivadas
Son columnas calculadas con una fórmula simple sobre una o pocas columnas existentes.

#### `dias_desde_pedido`

In [8]:
df["dias_desde_pedido"] = (fecha_corte - df["fecha_pedido"]).dt.days

In [9]:
df["dias_desde_pedido"].describe()

count    583003.000000
mean        823.090509
std         179.151562
min           1.000000
25%         704.000000
50%         834.000000
75%         963.000000
max        1092.000000
Name: dias_desde_pedido, dtype: float64

#### `gasto_por_unidad`
Se protege contra división por cero: si `unidades_vendidas` es 0, se deja como nulo.

In [10]:
df["gasto_por_unidad"] = df["monto_compra"] / df["unidades_vendidas"].replace(0, pd.NA)

In [11]:
df["gasto_por_unidad"].describe()

count     572880.0
unique      8771.0
top            0.0
freq       18208.0
Name: gasto_por_unidad, dtype: float64

#### `antiguedad_cliente_dias`
Días entre la fecha de inscripción al club y la fecha de corte. Queda nulo para clientes que no están en el club.

In [12]:
df["antiguedad_cliente_dias"] = (fecha_corte - df["fecha_inscripcion_club"]).dt.days

In [13]:
df["antiguedad_cliente_dias"].describe()

count    397933.000000
mean       1095.652690
std         421.776179
min         366.000000
25%         731.000000
50%        1096.000000
75%        1463.000000
max        1823.000000
Name: antiguedad_cliente_dias, dtype: float64

#### `dominio_correo`
Se extrae la parte del correo después del `@`.

In [14]:
df["dominio_correo"] = df["correo_cliente"].str.split("@").str[1]

In [15]:
df["dominio_correo"].value_counts(dropna=False).head(10)

dominio_correo
NaN                  146769
gmail.com             90626
outlook.com           90546
novamarket.com.co     90479
yahoo.es              90441
hotmail.com           90139
Name: count, dtype: int64

### Familia 2: Agregaciones por cliente
Resumen de todas las compras de cada cliente en un solo número usando `groupby`.

In [16]:
resumen_cliente = df.groupby("id_cliente").agg(
    numero_compras=("id_pedido", "count"),
    total_gastado=("monto_compra", "sum"),
    gasto_promedio=("monto_compra", "mean"),
    dias_desde_ultima_compra=("dias_desde_pedido", "min")
).reset_index()

In [17]:
resumen_cliente.head()

,id_cliente,numero_compras,total_gastado,gasto_promedio,dias_desde_ultima_compra
0,C10000,7,4419400.0,631342.857143,837.0
1,C100000,8,2564800.0,320600.000000,683.0
2,C100001,4,1306500.0,326625.000000,117.0
3,C100002,7,3197000.0,456714.285714,361.0
4,C100003,4,3768750.0,942187.500000,235.0


In [18]:
resumen_cliente.describe()

,numero_compras,total_gastado,gasto_promedio,dias_desde_ultima_compra
count,151762.000000,1.517620e+05,1.517620e+05,151415.000000
mean,3.946970,2.466592e+06,6.249254e+05,660.812707
std,1.903849,1.600727e+06,3.153605e+05,192.118703
min,1.000000,0.000000e+00,0.000000e+00,1.000000
25%,3.000000,1.231500e+06,3.984000e+05,613.000000
50%,4.000000,2.221500e+06,5.893333e+05,667.000000
75%,5.000000,3.406650e+06,8.037450e+05,756.000000
max,14.000000,1.375155e+07,1.809450e+06,1092.000000


### Familia 3: Features de dominio

Se construye el **puntaje RFM** , Ccda cliente recibe un puntaje de 1 a 4 en cada dimensión; sumados, dan un `rfm_score` entre 3 y 12.

In [21]:
# R: menos días desde última compra = mejor puntaje
resumen_cliente["score_R"] = 4 - pd.qcut(
    resumen_cliente["dias_desde_ultima_compra"], q=4, labels=False
)

In [22]:
# F: más compras = mejor puntaje
resumen_cliente["score_F"] = pd.qcut(
    resumen_cliente["numero_compras"].rank(method="first"), q=4, labels=False
) + 1

In [23]:
# M: más gasto = mejor puntaje
resumen_cliente["score_M"] = pd.qcut(
    resumen_cliente["total_gastado"], q=4, labels=False
) + 1

In [24]:
resumen_cliente["rfm_score"] = resumen_cliente["score_R"] + resumen_cliente["score_F"] + resumen_cliente["score_M"]

In [25]:
resumen_cliente["rfm_score"].describe()

count    151415.000000
mean          7.513542
std           2.644283
min           3.000000
25%           5.000000
50%           8.000000
75%          10.000000
max          12.000000
Name: rfm_score, dtype: float64

#### `segmento_rfm`
Se traduce el puntaje numérico a una etiqueta de negocio.

In [26]:
def clasificar_rfm(score):
    if score >= 10:
        return "Campeón"
    elif score >= 7:
        return "Leal"
    elif score >= 5:
        return "En riesgo"
    else:
        return "Perdido"

resumen_cliente["segmento_rfm"] = resumen_cliente["rfm_score"].apply(clasificar_rfm)
resumen_cliente["segmento_rfm"].value_counts()

segmento_rfm
Leal         54359
Campeón      40929
En riesgo    30957
Perdido      25517
Name: count, dtype: int64

#### `es_cliente_club`
Indica si el cliente tiene fecha de inscripción al club (True) o no (False).

In [27]:
df["es_cliente_club"] = df["fecha_inscripcion_club"].notnull()

In [28]:
df["es_cliente_club"].value_counts()

es_cliente_club
True     397933
False    201067
Name: count, dtype: int64

### Integración de las features de cliente al dataset principal
Se une el `resumen_cliente` al `df` por `id_cliente` para que cada pedido tenga, además de sus columnas originales, las features de su cliente.

In [29]:
df = df.merge(resumen_cliente, on="id_cliente", how="left")
df.shape

(599000, 78)

## Verificación final de las nuevas features

In [30]:
features_nuevas = [
    "dias_desde_pedido", "gasto_por_unidad", "antiguedad_cliente_dias", "dominio_correo",
    "numero_compras", "total_gastado", "gasto_promedio", "dias_desde_ultima_compra",
    "score_R", "score_F", "score_M", "rfm_score", "segmento_rfm", "es_cliente_club"
]
df[features_nuevas].head()

,dias_desde_pedido,gasto_por_unidad,antiguedad_cliente_dias,dominio_correo,numero_compras,total_gastado,gasto_promedio,dias_desde_ultima_compra,score_R,score_F,score_M,rfm_score,segmento_rfm,es_cliente_club
0,919.0,432300.0,NaN,NaN,5,2863600.0,5.727200e+05,636.0,3.0,3,3,9.0,Leal,False
1,809.0,582200.0,NaN,NaN,5,5632250.0,1.126450e+06,662.0,3.0,4,4,11.0,Campeón,False
2,942.0,234200.0,1241.0,outlook.com,3,591000.0,1.970000e+05,659.0,3.0,2,1,6.0,En riesgo,True
3,959.0,<NA>,748.0,yahoo.es,3,900300.0,3.001000e+05,596.0,4.0,2,1,7.0,Leal,True
4,1083.0,526600.0,NaN,NaN,3,1033600.0,3.445333e+05,663.0,3.0,2,1,6.0,En riesgo,False


In [31]:
df.shape

(599000, 78)

## Exportación del dataset con features

In [32]:
df.to_csv("S13_PD_Camacho_Serpa_novamarket_con_features.csv", index=False)